# PyTorch Foundations for Machine Learning

PyTorch implementations of core machine-learning operations, including tensor-based affine transformations, autograd, ridge regression, softmax classification, and PCA.

> Portfolio-ready copy of the original completed experiment. Experimental code and saved outputs are preserved; assignment-administration text has been removed.


In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
torch.manual_seed(0)
print("torch", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

torch 2.8.0+cu126


'cuda'

## Read first: Affine map, autograd, and broadcasting in PyTorch
A fully‑connected layer computes
$$
Y = XW + \mathbf{1}\, b^\top
$$
with $X\in\mathbb{R}^{N\times D}$, $W\in\mathbb{R}^{D\times K}$, $b\in\mathbb{R}^{K}$, $Y\in\mathbb{R}^{N\times K}$.  
PyTorch **broadcasts** `b` with shape `(K,)` across the batch. Autograd tracks ops to compute gradients via backprop.


## Exercise 1 (PyTorch) — Affine forward + autograd sanity
**a)** Create `X (N,D)`, `W (D,K)`, `b (K,)` as `torch.float32` on `device`.  
**b)** Compute three equivalent forwards:
- `Y1 = X @ W + b`
- `Y2 = torch.matmul(X, W) + b`
- `Y3 = torch.einsum('nd,dk->nk', X, W) + b`

Verify `torch.allclose` equality.  
**c)** Add a tiny target `Y_tgt` and compute MSE loss. Use autograd to compute `dL/dW` and `dL/db`; print their shapes.


In [4]:

# implement Exercise 1 (PyTorch)
N, D, K = 3, 3, 2
X = torch.tensor([[2.,-1.,0.5],[0.,1.,-1.],[1.,2.,3.]], device=device)
W = torch.tensor([[1.,0.],[2.,-1.],[-1.,1.]], device=device, requires_grad=True)
b = torch.tensor([0.5, -0.5], device=device, requires_grad=True)

# FIRST- Three equivalent affine forwards for Y1, Y2 and Y3 for verification purpose
Y1 = X @ W + b
Y2 = torch.matmul(X, W) + b
Y3 = torch.einsum('nd,dk->nk', X, W) + b

print(torch.allclose(Y1, Y2), torch.allclose(Y1, Y3))


# SECOND - loss/autograd: Define target and compute MSE loss
Y_tgt = torch.zeros_like(Y1)
loss = F.mse_loss(Y1, Y_tgt)
print("Loss =", loss.item())

# THIRD - Backpropagation
loss.backward()
print("dL/dW shape:", W.grad.shape, "dL/db shape:", b.grad.shape)

# Reset grads
W.grad.zero_(); b.grad.zero_()

True True
Loss = 4.333333492279053
dL/dW shape: torch.Size([3, 2]) dL/db shape: torch.Size([2])


tensor([0., 0.], device='cuda:0')

## Read first: Ridge regression two ways (closed form & weight decay)
Closed form (normal equation with L2):
$$
w_\lambda = (X^\top X + \lambda I)^{-1} X^\top y.
$$
In PyTorch training loops, **ridge** appears as **weight decay** (L2 penalty) in the optimizer (e.g., `optim.SGD(..., weight_decay=λ)`).


## Exercise 2 (PyTorch) — Ridge regression
Data (same as NumPy part): features in different scales.

**a)** Standardize each feature of `X_raw` (mean 0, std 1).  
**b)** Add bias column (ones) to get `X ∈ R^{N×(D+1)}`.  
**c)** **Closed form**: for `λ ∈ {0, 0.1, 1.0}`, compute
$$
w_\lambda = (X^\top X + \lambda I)^{-1} X^\top y
$$
using `torch.linalg.solve` (avoid explicit inverse). Report weights and MSE.  
**(Optional d)** Mini GD with weight decay: fit a `nn.Linear(D,1)` (with `bias=True`), use `MSELoss` and `optim.SGD(lr=0.1, weight_decay=λ)`. Compare MSE to closed form.


In [5]:
# implement Exercise 2 (PyTorch)
X_raw = torch.tensor([
    [3., 70., 20.],
    [2., 50., 40.],
    [4., 90., 15.],
    [3., 60., 30.],
    [5.,120., 10.],
    [1., 45., 50.],
    [2., 55., 35.],
    [4.,100., 12.]
], device=device)

y = torch.tensor([210.,160.,280.,200.,350.,120.,170.,320.], device=device).unsqueeze(1)

# FIRST - Standardize -> Xs
mu = X_raw.mean(dim=0, keepdim=True)
std = X_raw.std(dim=0, unbiased=False, keepdim=True)
Xs = (X_raw - mu) / torch.where(std==0, torch.ones_like(std), std)

# SECOND - Add bias -> X
N, D = Xs.shape
X = torch.cat([Xs, torch.ones(N,1, device=device)], dim=1)

# THIRD - Closed form Ridge Regression
XtX = X.T @ X
Xty = X.T @ y
R = torch.eye(D+1, device=device)
R[-1, -1] = 0.0

def ridge_closed_form(lam):
    A = XtX + lam * R
    w = torch.linalg.solve(A, Xty)
    yhat = X @ w
    mse = torch.mean((yhat - y)**2)
    return w, mse

lams = [0.0, 0.1, 1.0]
print('lambda\t' + '\t'.join([f'w{i}' for i in range(D)]) + '\tbias\tMSE')
for lam in lams:
    w, mse = ridge_closed_form(lam)
    w_flat = w.squeeze(1)
    row = [lam] + [w_flat[i].item() for i in range(D)] + [w_flat[-1].item(), mse.item()]
    print('\t'.join(f'{v: .4f}' for v in row))

# FORTH - Gradient Descent with weight decay
lam = 0.1
model = nn.Linear(3, 1, bias=True).to(device)
with torch.no_grad():
    model.weight.zero_(); model.bias.zero_()

opt = optim.SGD(model.parameters(), lr=0.1, weight_decay=lam)
loss_fn = nn.MSELoss()

for _ in range(50):
    opt.zero_grad()
    pred = model(Xs)
    loss = loss_fn(pred, y)
    loss.backward()
    opt.step()

print("Output for the Optional SGD (say λ=0.1) Then MSE:", loss.item())

lambda	w0	w1	w2	bias	MSE
 0.0000	 17.2547	 49.8051	-10.0299	 226.2500	 74.2250
 0.1000	 19.5301	 45.8548	-11.3360	 226.2500	 75.7487
 1.0000	 22.9005	 33.2569	-17.6290	 226.2500	 110.1112
Output for the Optional SGD (say λ=0.1) Then MSE: 213.758544921875


**My Observation:** It looks like Closed-form is exact and stable but the other way, weight Decay requires careful tuning.


## Read first: Multiclass softmax with `nn.Linear` + `CrossEntropyLoss`
`nn.CrossEntropyLoss` expects **raw logits** and **class indices** (not one‑hot). It applies `log_softmax` internally and computes NLL loss.


## Exercise 3 (PyTorch) — Softmax classifier w/ one GD step
**a)** Create 3 Gaussian blobs in 2‑D (same centers as NumPy part), stack to `(N,2)`.  
**b)** Build `model = nn.Linear(2, 3)` and `loss_fn = nn.CrossEntropyLoss()`.  
**c)** Compute loss/accuracy **before** training.  
**d)** Do **one** SGD step (e.g., `lr=0.5`), recompute loss/accuracy and report the change.


In [6]:

# implement Exercise 3 (PyTorch)
torch.manual_seed(1)
N_per, C = 50, 3
means = torch.tensor([[0.,0.],[2.5,2.5],[-2.5,2.5]], device=device)
X_parts = [torch.randn(N_per,2, device=device) + m for m in means]
X = torch.vstack(X_parts)                      # (150,2)
y_cls = torch.arange(C, device=device).repeat_interleave(N_per)  # (150,)

# Model: linear classifier (2 -> 3)
model = nn.Linear(2, 3).to(device)
loss_fn = nn.CrossEntropyLoss()

# Need to compute the initial loss & accuracy
with torch.no_grad():
    logits = model(X)
    loss0 = loss_fn(logits, y_cls).item()
    acc0 = (logits.argmax(dim=1) == y_cls).float().mean().item()

# Optimizer
optimizer = optim.SGD(model.parameters(), lr=0.5)

# One update step
optimizer.zero_grad()
loss = loss_fn(model(X), y_cls)
loss.backward()
optimizer.step()

# Loss & accuracy after one step
logits1 = model(X)
loss1 = loss_fn(logits1, y_cls).item()
acc1 = (logits1.argmax(dim=1) == y_cls).float().mean().item()

print(f"Loss before: {loss0:.4f}, acc: {acc0:.3f}")
print(f"Loss after : {loss1:.4f}, acc: {acc1:.3f}")

Loss before: 0.8273, acc: 0.673
Loss after : 0.6295, acc: 0.753


**My Observation-Reporting the Change: Change summary:**

Loss decreased by about 0.20 (≈24% improvement).

Accuracy increased by 8 percentage points.

This shows that even one gradient descent step significantly improves the model’s fit to the 3-class dataset.


## Read first: PCA in PyTorch (`torch.pca_lowrank` or `torch.linalg.svd`)
For standardized `X_std ∈ R^{N×D}`, PCA finds orthogonal directions of maximal variance.


## Exercise 4 (PyTorch) — PCA via `pca_lowrank`
**a)** Standardize `X_raw` (from Exercise 2).  
**b)** Use `torch.pca_lowrank(X_std, q=2)` (or SVD) to get top‑2 directions.  
**c)** Project `Z = X_std @ U2` and print shapes.  
**d)** (Optional) Compute explained variance ratio using singular values.


In [8]:
# implement Exercise 4 (PyTorch)
# Reuse X_raw from Exercise 2
# a) Standardize X_raw (from Exercise 2)
X_std = (X_raw - X_raw.mean(0)) / X_raw.std(0, unbiased=False)

# b) Use torch.pca_lowrank(X_std, q=2) to get top-2 directions
U, S, V = torch.pca_lowrank(X_std, q=2)  # V has shape (D, 2)

# c) Project onto top-2 directions
U2 = V[:, :2]
Z = X_std @ U2
print("U2 shape:", U2.shape, "Z shape:", Z.shape)

# d) (Optional) Explained variance ratio using singular values
#explained = (S[:2]**2).sum() / (S**2).sum()
#print(f"Explained variance (top-2): {explained.item()*100:.2f}%")

explained_ratios = (S**2) / (S**2).sum()
explained_pc1 = explained_ratios[0].item() * 100
explained_pc2 = explained_ratios[1].item() * 100
explained_top2 = (S[:2]**2).sum() / (S**2).sum() * 100

print(f"Explained variance PC1: {explained_pc1:.2f}%")
print(f"Explained variance PC2: {explained_pc2:.2f}%")
print(f"Explained variance (top-2): {explained_top2:.2f}%")

U2 shape: torch.Size([3, 2]) Z shape: torch.Size([8, 2])
Explained variance PC1: 98.04%
Explained variance PC2: 1.96%
Explained variance (top-2): 100.00%
